
# Bronze layer
- Silver for `investor` (internal CSV, 50 rows, dimension table).


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

In [0]:
bronze_df = read_bronze(spark, "investor")
print(f"Bronze row count: {bronze_df.count()}")

Bronze row count: 250


### Type casting & text standardization

In [0]:
typed_df = (
    bronze_df
    .withColumn("investor_id", F.trim(F.col("investor_id")))
    .withColumn("investor_name", F.trim(F.col("investor_name")))
    .withColumn("investor_type", F.trim(F.col("investor_type")))
    .withColumn("country", F.initcap(F.trim(F.col("country"))))
)

### Null check - all 4 columns required on this dimension table

In [0]:
REQUIRED_COLS = ["investor_id", "investor_name", "investor_type", "country"]
clean_df, null_rejects_df = split_on_required_nulls(typed_df, REQUIRED_COLS)
null_reject_count = null_rejects_df.count()
if null_reject_count > 0:
    write_quarantine(null_rejects_df, "investor")
print(f"Rows failing null check: {null_reject_count}")

Rows failing null check: 0


### Duplicate / break check - key = investor_id

In [0]:
KEY_COLS = ["investor_id"]
COMPARE_COLS = ["investor_name", "investor_type", "country"]
deduped_df, duplicates_df, breaks_df = split_duplicates(clean_df, KEY_COLS, COMPARE_COLS)

dup_count = duplicates_df.count()
break_count = breaks_df.count()
if dup_count > 0:
    write_quarantine(duplicates_df, "investor")
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("INVESTOR_ATTRIBUTE_BREAK")), "investor")
print(f"Exact duplicates: {dup_count} | Conflicting-value breaks: {break_count}")

Exact duplicates: 200 | Conflicting-value breaks: 0


### Write to Silver + log DQ

In [0]:
write_silver(deduped_df, "investor")

business_date_str = date.today().isoformat()
log_dq(spark, "investor", business_date_str, "null_required_field", bronze_df.count(), null_reject_count, "NULL_REQUIRED_FIELD")
log_dq(spark, "investor", business_date_str, "duplicate_record", clean_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, "investor", business_date_str, "attribute_break", clean_df.count(), break_count, "INVESTOR_ATTRIBUTE_BREAK")

/home/spark-4c42cdef-a13f-495c-ad7c-9e/.ipykernel/70/command-5696143635338715-3986853518:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


### Sanity check

In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = null_reject_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=250 = silver=50 + quarantined=200
